## Homework: Futures Spread Dynamics

## Data Preprocessing

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import databento as db
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from joblib import Parallel, delayed
from plotly.subplots import make_subplots

### Configuration constants

In [2]:
DATA_DIRECTORY: Path = Path(
    "/Users/dipesh/Desktop/QTS/data/CME_1minute_Dec_2025"
)

START_DATE: str = "20251212"
END_DATE: str = "20251226"

ROOT_SYMBOLS: Tuple[str, ...] = ("GC", "SIL", "CL", "HO")

JANUARY_2026_SUFFIX: str = "F6"
FEBRUARY_2026_SUFFIX: str = "G6"
MARCH_2026_SUFFIX: str = "H6"

### File handling utilities

In [3]:
def generate_trading_dates(start_date: str, end_date: str) -> List[str]:
    """
    Generate YYYYMMDD strings for each calendar day in the range.

    Parameters
    ----------
    start_date : str
        Start date in YYYYMMDD format.
    end_date : str
        End date in YYYYMMDD format.

    Returns
    -------
    List[str]
        List of date strings.
    """
    return (
        pd.date_range(start_date, end_date, freq="D")
        .strftime("%Y%m%d")
        .tolist()
    )
    
    
def load_daily_ohlcv(file_path: Path) -> pd.DataFrame:
    """
    Load a single Databento DBN OHLCV file into a pandas DataFrame.
    
    Parameters
    ----------
    file_path : Path
        Path to the .dbn.zst file.

    Returns
    -------
    pd.DataFrame
        Raw OHLCV data with timestamps converted.
    """
    store: db.DBNStore = db.DBNStore.from_file(file_path)
    dataframe: pd.DataFrame = store.to_df()

    return dataframe

### Symbol filtering logic

In [4]:
def is_outright_contract(symbol: str) -> bool:
    """
    Determine whether a symbol represents an outright futures contract.

    Excludes:
    - Calendar spreads (contain '-')
    - Strategy/synthetic symbols (contain ':')

    Parameters
    ----------
    symbol : str
        Symbol string.

    Returns
    -------
    bool
        True if outright contract, False otherwise.
    """
    return "-" not in symbol and ":" not in symbol


def filter_relevant_symbols(
    dataframe: pd.DataFrame,
    root_symbols: Iterable[str],
) -> pd.DataFrame:
    """
    Filter DataFrame to outright contracts for specified root symbols only.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Raw OHLCV data.
    root_symbols : Iterable[str]
        Root symbols to keep (e.g., GC, SIL).

    Returns
    -------
    pd.DataFrame
        Filtered DataFrame.
    """
    starts_with_root = dataframe["symbol"].str.startswith(tuple(root_symbols))
    is_outright = dataframe["symbol"].apply(is_outright_contract)

    return dataframe[starts_with_root & is_outright].copy()

### Contract selection logic

For each futures symbol (GC, SIL, CL, HO), the front and second month
contracts are selected based on **contract availability and trading activity**,
not simply the calendar month.

During the sample period (December 2025), several contracts with a December
expiry (`Z5`) exhibited **substantially lower trading activity** relative to
the January 2026 contracts. In particular, the January contracts (`F6`) had
both:
- Higher cumulative traded volume, and
- A larger number of 1-minute OHLCV observations (data points)

As a result, the January contract represented the **most actively traded
contract** during the sample window and was therefore treated as the effective
front month.

The selection logic is:

- **If the January 2026 contract (`F6`) exists** in the dataset:
  - Front month = `F6`
  - Second month = `G6`
- **Otherwise**:
  - Front month = `G6`
  - Second month = `H6`

This rule is applied **independently for each symbol** and **for each
trading day**, ensuring that the front month corresponds to the **nearest
actively traded expiration** present in the data rather than the nominal
calendar expiry.


In [5]:
def select_front_and_second_contracts(
    dataframe: pd.DataFrame,
    root_symbol: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Select front and second month contracts for a given root symbol.

    Logic:
    - If January 2026 (F6) exists → front = F6, second = G6
    - Else → front = G6, second = H6

    Parameters
    ----------
    dataframe : pd.DataFrame
        Filtered OHLCV data.
    root_symbol : str
        Root symbol (e.g., GC).

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame]
        (front_month_df, second_month_df)

    Raises
    ------
    ValueError
        If required contracts are missing.
    """
    available_symbols: set[str] = set(dataframe["symbol"].unique())

    january_contract: str = f"{root_symbol}{JANUARY_2026_SUFFIX}"
    february_contract: str = f"{root_symbol}{FEBRUARY_2026_SUFFIX}"
    march_contract: str = f"{root_symbol}{MARCH_2026_SUFFIX}"

    if january_contract in available_symbols:
        front_symbol = january_contract
        second_symbol = february_contract
    else:
        front_symbol = february_contract
        second_symbol = march_contract

    front_dataframe = dataframe[dataframe["symbol"] == front_symbol].copy()
    second_dataframe = dataframe[dataframe["symbol"] == second_symbol].copy()

    if front_dataframe.empty or second_dataframe.empty:
        raise ValueError(
            f"Missing required contracts for {root_symbol}: "
            f"{front_symbol}, {second_symbol}"
        )

    return front_dataframe, second_dataframe

### Main aggregation routine

In [6]:
def _process_single_date(date_string: str) -> Dict[str, List[pd.DataFrame]]:
    """
    Process a single trading date and return contract data for all root symbols.
    
    Parameters
    ----------
    date_string : str
        Trading date string.
    
    Returns
    -------
    Dict[str, List[pd.DataFrame]]
        Dictionary with keys like 'GC1', 'GC2', etc., each containing a list
        with a single DataFrame (or empty list if file doesn't exist).
    """
    file_path: Path = (
        DATA_DIRECTORY / f"glbx-mdp3-{date_string}.ohlcv-1m.dbn.zst"
    )
    
    result: Dict[str, List[pd.DataFrame]] = {
        "GC1": [],
        "GC2": [],
        "SIL1": [],
        "SIL2": [],
        "CL1": [],
        "CL2": [],
        "HO1": [],
        "HO2": [],
    }
    
    if not file_path.exists():
        return result
    
    daily_data: pd.DataFrame = load_daily_ohlcv(file_path)
    daily_data = filter_relevant_symbols(
        daily_data, ROOT_SYMBOLS
    )
    
    for root_symbol in ROOT_SYMBOLS:
        front_df, second_df = select_front_and_second_contracts(
            daily_data, root_symbol
        )
        
        result[f"{root_symbol}1"].append(front_df)
        result[f"{root_symbol}2"].append(second_df)
    
    return result


def build_contract_time_series() -> Dict[str, pd.DataFrame]:
    """
    Load, process, and aggregate front/second month futures data
    for all root symbols across the full date range.

    Returns
    -------
    Dict[str, pd.DataFrame]
        Dictionary containing eight DataFrames:
        GC1, GC2, SIL1, SIL2, CL1, CL2, HO1, HO2
    """
    
    trading_dates: List[str] = list(generate_trading_dates(START_DATE, END_DATE))
    
    # Process all dates in parallel
    results: List[Dict[str, List[pd.DataFrame]]] = Parallel(n_jobs=-1)(
        delayed(_process_single_date)(date_string)
        for date_string in trading_dates
    )
    
    # Aggregate results from all dates
    aggregated_data: Dict[str, List[pd.DataFrame]] = {
        "GC1": [],
        "GC2": [],
        "SIL1": [],
        "SIL2": [],
        "CL1": [],
        "CL2": [],
        "HO1": [],
        "HO2": [],
    }
    
    for daily_result in results:
        for key in aggregated_data.keys():
            aggregated_data[key].extend(daily_result[key])
    
    return {
        key: (
            pd.concat(dataframes)
        )
        for key, dataframes in aggregated_data.items()
    }

### Get front month and second month contract

In [7]:
contract_series: Dict[str, pd.DataFrame] = build_contract_time_series()

GC1: pd.DataFrame = contract_series["GC1"]
GC2: pd.DataFrame = contract_series["GC2"]
GC = GC1[["close"]].merge(GC2[["close"]], how="outer", left_index=True, right_index=True, suffixes=('_GC1', '_GC2')).ffill().dropna()

SIL1: pd.DataFrame = contract_series["SIL1"]
SIL2: pd.DataFrame = contract_series["SIL2"]
SIL = SIL1[["close"]].merge(SIL2[["close"]], how="outer", left_index=True, right_index=True, suffixes=('_SIL1', '_SIL2')).ffill().dropna()

CL1: pd.DataFrame = contract_series["CL1"]
CL2: pd.DataFrame = contract_series["CL2"]
CL = CL1[["close"]].merge(CL2[["close"]], how="outer", left_index=True, right_index=True, suffixes=('_CL1', '_CL2')).ffill().dropna()

HO1: pd.DataFrame = contract_series["HO1"]
HO2: pd.DataFrame = contract_series["HO2"]
HO = HO1[["close"]].merge(HO2[["close"]], how="outer", left_index=True, right_index=True, suffixes=('_HO1', '_HO2')).ffill().dropna()

### Create calendar spreads 

For each futures contract, the calendar spread is defined as the difference
between the **second month** price and the **front month** price of the same
underlying futures contract.

Formally, for each asset:

$$
s^{(GC)}_t = \text{GC}_{\text{second}}(t) - \text{GC}_{\text{front}}(t)
$$

$$
s^{(SIL)}_t = \text{SIL}_{\text{second}}(t) - \text{SIL}_{\text{front}}(t)
$$

$$
s^{(CL)}_t = \text{CL}_{\text{second}}(t) - \text{CL}_{\text{front}}(t)
$$

$$
s^{(HO)}_t = \text{HO}_{\text{second}}(t) - \text{HO}_{\text{front}}(t)
$$


In [8]:
# Difference between second and front month contract prices
GC['calendar_spread'] = GC['close_GC2'] - GC['close_GC1']
SIL['calendar_spread'] = SIL['close_SIL2'] - SIL['close_SIL1']
CL['calendar_spread'] = CL['close_CL2'] - CL['close_CL1']
HO['calendar_spread'] = HO['close_HO2'] - HO['close_HO1']

## Analysis

In [9]:
# Define symbols and their display names for plotting
symbols = [
    ("GC", "Gold"),
    ("SIL", "Silver"),
    ("CL", "Crude Oil"),
    ("HO", "Heating Oil"),
]

# Create subplot titles for each symbol's front and second month contracts
subplot_titles = []
for symbol_code, symbol_name in symbols:
    subplot_titles.append(f"{symbol_name} Front Month ({symbol_code}1) Close Price")
    subplot_titles.append(f"{symbol_name} Second Month ({symbol_code}2) Close Price")

# Initialize figure with 4 rows (one per symbol) and 2 columns (front/second month)
fig = make_subplots(
    rows=4,
    cols=2,
    subplot_titles=tuple(subplot_titles),
    vertical_spacing=0.08,
    horizontal_spacing=0.1,
)

# Define colors for front month and second month contracts
colors = ["#2E86AB", "#A23B72"]

# Plot close prices for each symbol across both tenors
for row_idx, (symbol_code, symbol_name) in enumerate(symbols, start=1):
    # Retrieve front month and second month dataframes
    front_month = contract_series[f"{symbol_code}1"]
    second_month = contract_series[f"{symbol_code}2"]

    # Plot front month contract (column 1)
    front_reset = front_month.reset_index()
    fig.add_trace(
        go.Scatter(
            x=front_reset["ts_event"],
            y=front_reset["close"],
            mode="lines",
            line=dict(color=colors[0], width=1),
            name=f"{symbol_code}1",
        ),
        row=row_idx,
        col=1,
    )

    # Plot second month contract (column 2)
    second_reset = second_month.reset_index()
    fig.add_trace(
        go.Scatter(
            x=second_reset["ts_event"],
            y=second_reset["close"],
            mode="lines",
            line=dict(color=colors[1], width=1),
            name=f"{symbol_code}2",
        ),
        row=row_idx,
        col=2,
    )

    # Update axis labels for both columns
    fig.update_xaxes(title_text="Date", row=row_idx, col=1)
    fig.update_xaxes(title_text="Date", row=row_idx, col=2)
    fig.update_yaxes(title_text="Close Price ($)", row=row_idx, col=1)
    fig.update_yaxes(title_text="Close Price ($)", row=row_idx, col=2)

# Update overall layout with title and dimensions
fig.update_layout(
    title={
        "text": "Close Prices Across Tenors",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20},
    },
    height=1500,
    width=1350,
    showlegend=True,
)

fig.show()

In [10]:
# Define symbols and their display names for plotting
symbols = [
    ("GC", "Gold"),
    ("SIL", "Silver"),
    ("CL", "Crude Oil"),
    ("HO", "Heating Oil"),
]

# Create subplot titles for each symbol's calendar spread
subplot_titles = []
for symbol_code, symbol_name in symbols:
    subplot_titles.append(f"{symbol_name} ({symbol_code}) Calendar Spread")

# Initialize figure with 2 rows and 2 columns
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=tuple(subplot_titles),
    vertical_spacing=0.15,
    horizontal_spacing=0.1,
)

# Define color for calendar spreads
spread_color = "#2E86AB"

# Plot calendar spreads for each symbol
for idx, (symbol_code, symbol_name) in enumerate(symbols, start=1):
    # Calculate row and column position (2x2 grid)
    row = ((idx - 1) // 2) + 1
    col = ((idx - 1) % 2) + 1
    
    # Retrieve the merged dataframe for this symbol
    symbol_df = locals()[symbol_code]
    
    # Reset index to access timestamp
    symbol_reset = symbol_df.reset_index()
    
    # Use index column if it exists, otherwise use the reset index
    x_values = symbol_reset.index if "ts_event" not in symbol_reset.columns else symbol_reset["ts_event"]
    
    # Plot calendar spread
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=symbol_reset["calendar_spread"],
            mode="lines",
            line=dict(color=spread_color, width=1.5),
            name=f"{symbol_code} Spread",
        ),
        row=row,
        col=col,
    )
    
    # Update axis labels
    fig.update_xaxes(title_text="Date", row=row, col=col)
    fig.update_yaxes(title_text="Spread ($)", row=row, col=col)

# Update overall layout with title and dimensions
fig.update_layout(
    title={
        "text": "Calendar Spreads Across Futures Assets",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20},
    },
    height=850,
    width=1350,
    showlegend=True,
)

fig.show()

Observing the calendar spreads across the sample, we notice a key difference between gold and the other assets. For gold, the spread remains strictly positive at all times, indicating that the price of the second month futures contract was consistently higher than that of the front month. This persistent positive spread shows that gold was continuously in contango, reflecting a higher and more stable cost of carry compared to the other assets. In contrast, the spreads for the other symbols oscillate from positive to negative values throughout the period. This means that, unlike gold, these assets experienced periods of both contango (second month > front month) and backwardation (front month > second month). Therefore, while most assets alternated between market states, gold stood out as having a uniquely persistent contango structure throughout the entire sample, highlighting its relatively higher and consistent cost of carry.

In [11]:
# Define symbols and their display names
symbols = [
    ("GC", "Gold"),
    ("SIL", "Silver"),
    ("CL", "Crude Oil"),
    ("HO", "Heating Oil"),
]

# Compute statistics for each calendar spread
statistics_data = []

for symbol_code, symbol_name in symbols:
    # Retrieve the merged dataframe for this symbol
    symbol_df = locals()[symbol_code]
    spread_data = symbol_df["calendar_spread"]
    
    # Compute statistics
    mean_value = spread_data.mean()
    median_value = spread_data.median()
    std_dev = spread_data.std()
    min_value = spread_data.min()
    max_value = spread_data.max()
    q1 = spread_data.quantile(0.25)
    q3 = spread_data.quantile(0.75)
    iqr = q3 - q1
    
    # Store statistics in list
    statistics_data.append({
        "Asset": f"{symbol_name} ({symbol_code})",
        "Mean": mean_value,
        "Median": median_value,
        "Std Dev": std_dev,
        "Min": min_value,
        "Max": max_value,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
    })

# Create and display statistics dataframe
stats_df = pd.DataFrame(statistics_data)
stats_df = stats_df.round(4)  # Round to 4 decimal places
display(stats_df)

# Create subplot titles for each symbol's box plot
subplot_titles = []
for symbol_code, symbol_name in symbols:
    subplot_titles.append(f"{symbol_name} ({symbol_code}) Calendar Spread")

# Initialize figure with 1 row and 4 columns
fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=tuple(subplot_titles),
    vertical_spacing=0.08,
    horizontal_spacing=0.10,
)

# Define color for box plots
box_color = "#2E86AB"

# Create box plots for each symbol in separate subplots (in 1 row, 4 columns)
for idx, (symbol_code, symbol_name) in enumerate(symbols, start=1):
    row = 1
    col = idx
    
    # Retrieve the merged dataframe for this symbol
    symbol_df = locals()[symbol_code]
    spread_data = symbol_df["calendar_spread"]
    
    # Add box plot to subplot
    fig.add_trace(
        go.Box(
            y=spread_data.values,
            name=f"{symbol_code}",
            boxmean="sd",  # Show mean and standard deviation
            boxpoints="outliers",  # Show outliers
            marker_color=box_color,
            line=dict(color=box_color, width=1.5),
        ),
        row=row,
        col=col,
    )
    
    # Update axis labels
    fig.update_xaxes(title_text="", row=row, col=col, showticklabels=False)
    fig.update_yaxes(title_text="Spread ($)", row=row, col=col)

# Update overall layout with title and dimensions
fig.update_layout(
    title={
        "text": "Calendar Spread Distribution - Box and Whisker Plots",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20},
    },
    height=500,
    width=1350,
    showlegend=False,
)

fig.show()

,Asset,Mean,Median,Std Dev,Min,Max,Q1,Q3,IQR
0,Gold (GC),17.3508,17.3000,2.1296,6.0000,29.3000,16.2000,18.4000,2.2000
1,Silver (SIL),0.2547,0.2600,0.1265,-0.3700,1.0200,0.1950,0.3150,0.1200
2,Crude Oil (CL),-0.1760,-0.1700,0.0493,-0.3300,0.1300,-0.2000,-0.1500,0.0500
3,Heating Oil (HO),-0.0071,-0.0075,0.0026,-0.0156,0.0065,-0.0088,-0.0056,0.0032


In [12]:
# Define symbols and their display names
symbols = [
    ("GC", "Gold"),
    ("SIL", "Silver"),
    ("CL", "Crude Oil"),
    ("HO", "Heating Oil"),
]

# Create subplot titles for each symbol's kernel density
subplot_titles = []
for symbol_code, symbol_name in symbols:
    subplot_titles.append(f"{symbol_name} ({symbol_code})")

# Initialize figure with 1 row and 4 columns, set margin bottom for extra space
fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=tuple(subplot_titles),
    vertical_spacing=0.15,
    horizontal_spacing=0.08,
)

# Define color for density plots
density_color = "#2E86AB"

# Plot kernel density for each symbol
for idx, (symbol_code, symbol_name) in enumerate(symbols, start=1):
    # Retrieve the merged dataframe for this symbol
    symbol_df = locals()[symbol_code]
    spread_data = symbol_df["calendar_spread"].values

    # Compute kernel density estimate
    kde = stats.gaussian_kde(spread_data)

    # Create range of x values for plotting
    x_min = spread_data.min()
    x_max = spread_data.max()
    x_range = np.linspace(x_min, x_max, 500)
    density_values = kde(x_range)

    # Calculate skewness and kurtosis
    skewness = stats.skew(spread_data)
    kurtosis = stats.kurtosis(spread_data)

    # Add density plot to subplot
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=density_values,
            mode="lines",
            line=dict(color=density_color, width=2),
            fill="tozeroy",
            fillcolor=f"rgba(46, 134, 171, 0.3)",  # Semi-transparent fill
            name=f"{symbol_code}",
            showlegend=False,
        ),
        row=1,
        col=idx,
    )

    # Add annotation with skewness and kurtosis centered under the plot area
    annotation_text = f"Skewness: {skewness:.4f}<br>Kurtosis: {kurtosis:.4f}"
    fig.add_annotation(
        text=annotation_text,
        xref=f"x{idx} domain" if idx > 1 else "x domain",
        yref="paper",
        x=0.5,
        y=-0.20,  # Move annotation further down, more negative = more space below plot
        xanchor="center",
        yanchor="top",
        showarrow=False,
        font=dict(size=12),
    )

    # Update axis labels
    fig.update_xaxes(title_text="Spread ($)", row=1, col=idx)
    fig.update_yaxes(title_text="Density", row=1, col=idx)

# Update overall layout with title, dimensions, and bottom margin for space
fig.update_layout(
    title={
        "text": "Kernel Density Estimates of Calendar Spreads",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20},
    },
    height=450,
    width=1350,
    showlegend=False,
    margin=dict(b=100),  # Increase bottom margin to preserve annotation space when height increases
)

fig.show()

In [42]:
def combine_all_calendar_spreads(
    gc_dataframe: pd.DataFrame,
    sil_dataframe: pd.DataFrame,
    cl_dataframe: pd.DataFrame,
    ho_dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Combine calendar spreads from all futures symbols into a single DataFrame.
    
    Extracts the calendar_spread column from each symbol's dataframe, renames
    them with the symbol identifier, and merges them on the datetime index using
    an outer join to preserve all timestamps.
    
    Parameters
    ----------
    gc_dataframe : pd.DataFrame
        Gold futures dataframe with calendar_spread column.
    sil_dataframe : pd.DataFrame
        Silver futures dataframe with calendar_spread column.
    cl_dataframe : pd.DataFrame
        Crude oil futures dataframe with calendar_spread column.
    ho_dataframe : pd.DataFrame
        Heating oil futures dataframe with calendar_spread column.
    
    Returns
    -------
    pd.DataFrame
        Combined dataframe with columns:
        - calendar_spread_GC: Gold calendar spread
        - calendar_spread_SIL: Silver calendar spread
        - calendar_spread_CL: Crude oil calendar spread
        - calendar_spread_HO: Heating oil calendar spread
        Index is datetime from all merged dataframes.
    """
    # Extract calendar spread columns and rename with symbol identifiers
    gc_spread: pd.Series = gc_dataframe["calendar_spread"].rename("calendar_spread_GC")
    sil_spread: pd.Series = sil_dataframe["calendar_spread"].rename("calendar_spread_SIL")
    cl_spread: pd.Series = cl_dataframe["calendar_spread"].rename("calendar_spread_CL")
    ho_spread: pd.Series = ho_dataframe["calendar_spread"].rename("calendar_spread_HO")
    
    # Combine all spreads into a single dataframe using outer join
    # This preserves all timestamps from all symbols
    combined_spreads: pd.DataFrame = pd.DataFrame({
        "calendar_spread_GC": gc_spread,
        "calendar_spread_SIL": sil_spread,
        "calendar_spread_CL": cl_spread,
        "calendar_spread_HO": ho_spread,
    })
    
    return combined_spreads


# Create the combined calendar spreads dataframe
all_calendar_spreads: pd.DataFrame = combine_all_calendar_spreads(
    gc_dataframe=GC,
    sil_dataframe=SIL,
    cl_dataframe=CL,
    ho_dataframe=HO,
)

corr_matrix = all_calendar_spreads.corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1,
    title='Correlation Heatmap for Calendar Spreads for different futures',
    labels=dict(x="Spread", y="Spread", color="Correlation"),
)
fig.update_layout(
    width=700, height=650,
    font=dict(size=14),
    xaxis=dict(tickfont=dict(size=13)),
    yaxis=dict(tickfont=dict(size=13)),
)
fig.show()